# 뉴스 사건 클러스터링 실험 A (Colab)

> 대상: **AI-reviewed reference set** (human gold set 아님)
> 실행 방법: 런타임을 **GPU(T4 이상)** 로 설정 → **Runtime ▸ Run all**
> ⚠️ 이 노트북은 **실행 전 템플릿**입니다(출력 미포함, #13). Run all 후 `results/`·`plots/`가 채워집니다.

임베딩 3종 × 입력 3형태 × **기본 클러스터링 3방법**(centroid/agglomerative/Leiden)을
validation에서 비교하고, B-cubed F1을 주지표(over-merge 낮은 순)로 최종 설정을 선택한 뒤
**test에서 1회만** 평가합니다. reranker+Leiden은 **선택적 확장 실험**으로 validation에서만
시연합니다(정식 선정/ test 미사용).

**split 역할(#3):**
- **development**: 알고리즘 개발·탐색 범위(threshold 등) 확인용. (이 노트북은 검증된 범위를
  바로 validation 스윕에 사용하므로 dev를 스윕에 넣지 않음 — 로컬 `run_experiment.py --split development`로 탐색 가능)
- **validation**: 모델·입력·알고리즘·threshold **최종 선택**.
- **test**: 확정 설정 **1회** 평가.
> split은 `split_unit_id` 단위로 나뉘어 같은 사건/공유 기사가 서로 다른 split에 새지 않습니다
> (누수 방지). 아래 셀에서 누수 검사도 수행합니다.


## 0. 업로드할 파일
왼쪽 파일 탭에 아래를 업로드하거나, 아래 셀에서 `files.upload()`를 쓰세요.
- `development.csv`, `validation.csv`, `test.csv` (experiments/splits/)
- (선택) `cluster_reference_ai_reviewed_final.csv`


In [ ]:
# 의존성 설치 (Colab)
!pip -q install sentence-transformers==3.* scikit-learn python-igraph leidenalg matplotlib pandas
# reranker (방법 4)용
!pip -q install FlagEmbedding 2>/dev/null || echo "FlagEmbedding 선택 설치 실패(무시 가능)"

In [ ]:
# 실행 환경 검사 (프롬프트 [3])
import subprocess
import sys

print("Python:", sys.version.split()[0])
try:
    print(subprocess.check_output(["nvidia-smi"]).decode()[:600])
except Exception as e:
    print("nvidia-smi 없음:", e)
import torch

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        "| VRAM(GB):",
        round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    )
print("CUDA(torch):", torch.version.cuda)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

Python: 3.12.13
Mon Jul 20 06:28:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===============
PyTorch: 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4 | VRAM(GB): 15.6
CUDA(torch): 12.8
DEVICE = cuda


In [ ]:
# 파일 업로드 (이미 올렸으면 건너뜀)
import os

need = [f for f in ["development.csv", "validation.csv", "test.csv"] if not os.path.exists(f)]
if need:
    from google.colab import files

    print("업로드하세요:", need)
    files.upload()
assert all(os.path.exists(f) for f in ["development.csv", "validation.csv", "test.csv"]), (
    "splits CSV 필요"
)
print("splits 준비 완료")

splits 준비 완료


## 1. 실험 라이브러리 (clustering_lib 인라인)
로컬 `experiments/clustering_lib.py`와 동일한 로직입니다.


In [ ]:
"""실험 A 공용 라이브러리: 입력 텍스트 구성, 임베딩(캐시), 클러스터링, 평가.

로컬(코드 검증)과 Colab(GPU 실행) 양쪽에서 동일하게 import해서 쓴다.
무거운 의존성(torch, sentence-transformers, igraph/leidenalg)은 함수 안에서 지연 import
하므로, 데이터/평가 유틸만 쓸 때는 설치 없이도 로드된다.
"""

from __future__ import annotations

import hashlib
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

# ---------------------------------------------------------------- 입력 텍스트
INPUT_TYPES = ("A_title", "B_title_desc", "C_title_desc_body")

# 전처리 로직 버전. build_input_text / format_for_model 을 바꾸면 이 값을 올려
# 기존 임베딩 캐시를 무효화한다(#2). 캐시 키에 포함된다.
PREPROCESS_VERSION = "v1"


def build_input_text(row: dict, input_type: str) -> str:
    """A: title / B: title+description / C: title+description+body_head.

    body_head가 비면 C는 자동으로 B와 동일해진다(프롬프트 규칙).
    """

    title = (row.get("title") or "").strip()
    desc = (row.get("description") or "").strip()
    body = (row.get("body_head") or "").strip()
    if input_type == "A_title":
        return title
    if input_type == "B_title_desc":
        return " ".join(p for p in (title, desc) if p)
    if input_type == "C_title_desc_body":
        parts = [title, desc] + ([body] if body else [])
        return " ".join(p for p in parts if p)
    raise ValueError(f"unknown input_type: {input_type}")


def text_sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# ---------------------------------------------------------------- 임베딩 캐시
@dataclass
class EmbeddingCache:
    """(model, revision, input_type, text_sha256) → vector 캐시.

    디스크에 npz+json으로 저장해 재실행 시 임베딩 재계산을 피한다.
    """

    cache_dir: Path
    _mem: dict[str, np.ndarray] = field(default_factory=dict)

    def key(
        self,
        model: str,
        revision: str,
        input_type: str,
        sha: str,
        preprocess_version: str = PREPROCESS_VERSION,
        max_seq_length: int | str = "default",
    ) -> str:
        """캐시 키. sha는 **전처리 완료(prepared) 텍스트**의 SHA-256를 넘겨야 한다.

        전처리(E5 instruction 등)나 max_seq_length가 바뀌면 키도 바뀌어야 하므로
        preprocess_version과 max_seq_length를 키에 포함한다(#2).
        """

        return (
            f"{model}||{revision}||{input_type}||{preprocess_version}||msl={max_seq_length}||{sha}"
        )

    def path(self, model: str) -> Path:
        safe = model.replace("/", "__")
        return self.cache_dir / f"emb_{safe}.npz"

    def load(self, model: str) -> None:
        p = self.path(model)
        if p.exists():
            data = np.load(p, allow_pickle=True)
            keys = list(data["keys"])
            vecs = data["vecs"]
            for k, v in zip(keys, vecs):
                self._mem[str(k)] = v

    def save(self, model: str) -> None:
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        prefix = f"{model}||"
        items = [(k, v) for k, v in self._mem.items() if k.startswith(prefix)]
        if not items:
            return
        keys = np.array([k for k, _ in items], dtype=object)
        vecs = np.vstack([v for _, v in items])
        np.savez_compressed(self.path(model), keys=keys, vecs=vecs)

    def get(self, key: str) -> np.ndarray | None:
        return self._mem.get(key)

    def put(self, key: str, vec: np.ndarray) -> None:
        self._mem[key] = vec


def l2_normalize(mat: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return mat / norms


# ---------------------------------------------------------------- 임베더
# 뉴스-뉴스 대칭 유사도용 instruction (E5-instruct 계열). 대칭 태스크이므로 모든
# 문장에 동일한 instruction을 붙인다(질의/문서 비대칭 아님).
_E5_INSTRUCT = "Instruct: Retrieve news articles that report the same news event\nQuery: "


def format_for_model(text: str, model_name: str) -> str:
    """모델별 권장 입력 프리픽스 적용 (#6).

    - e5-*-instruct: 대칭 유사도이므로 모든 문장에 동일 instruction 프리픽스.
    - e5 (non-instruct): 대칭 비교에는 'query:' 프리픽스를 양쪽에 동일 적용.
    - bge-m3: 프리픽스 없음(원문 그대로).
    """

    name = model_name.lower()
    t = text or ""
    if "e5" in name and "instruct" in name:
        return _E5_INSTRUCT + t
    if "e5" in name:
        return f"query: {t}"
    return t  # bge-m3 등


def embed_sentence_transformer(
    prepared_texts: list[str],
    model_name: str,
    revision: str | None,
    device: str,
    batch_size: int = 64,
    max_seq_length: int | None = None,
) -> tuple[np.ndarray, dict]:
    """BGE-M3 / E5 계열 임베딩.

    입력은 **이미 전처리(format_for_model)된 prepared 텍스트**여야 한다. 캐시 키가
    prepared 텍스트 SHA로 계산되므로, 임베딩 입력과 키가 정확히 일치하도록
    이 함수는 프리픽스를 다시 붙이지 않는다(#2). truncation 비율을 함께 보고(#6).

    반환: (vecs, meta).
    """

    from sentence_transformers import SentenceTransformer

    st = SentenceTransformer(model_name, revision=revision, device=device)
    if max_seq_length:
        st.max_seq_length = max_seq_length

    tok = st.tokenizer
    limit = st.max_seq_length
    lengths = [len(tok.encode(t, add_special_tokens=True)) for t in prepared_texts]
    truncated = sum(1 for n in lengths if n > limit)
    meta = {
        "max_seq_length": int(limit),
        "truncation_rate": round(truncated / len(prepared_texts), 4) if prepared_texts else 0.0,
        "max_token_len": int(max(lengths)) if lengths else 0,
        "resolved_revision": _resolve_st_revision(st, model_name, revision),
    }

    vecs = st.encode(
        prepared_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return vecs.astype(np.float32), meta


def embedding_meta_only(
    prepared_texts: list[str],
    model_name: str,
    revision: str | None,
    max_seq_length: int | None = None,
) -> dict:
    """임베딩 벡터 계산 없이 metadata(truncation 비율, max_token_len, revision)만 산출.

    100% cache hit이어도 truncation/revision 메타를 채우기 위해 사용(#3-테스트).
    토크나이저만 로드하므로 임베딩 인코딩보다 훨씬 가볍다.
    """

    from transformers import AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_name, revision=revision)
    limit = max_seq_length or getattr(tok, "model_max_length", 512)
    if limit is None or limit > 100000:
        limit = 512
    lengths = [len(tok.encode(t, add_special_tokens=True)) for t in prepared_texts]
    truncated = sum(1 for n in lengths if n > limit)
    return {
        "max_seq_length": int(limit),
        "truncation_rate": round(truncated / len(prepared_texts), 4) if prepared_texts else 0.0,
        "max_token_len": int(max(lengths)) if lengths else 0,
        "resolved_revision": revision or "main",
        "from_cache": True,
    }


def _resolve_st_revision(st, model_name: str, revision: str | None) -> str:
    """실제 로드된 모델의 commit hash를 최대한 회수(#5). 실패 시 요청 revision."""

    try:
        from huggingface_hub import HfApi

        info = HfApi().model_info(model_name, revision=revision or "main")
        return info.sha or (revision or "main")
    except Exception:  # noqa: BLE001
        return revision or "main"


def embed_upstage(texts: list[str], api_key: str, model: str = "embedding-passage") -> np.ndarray:
    """Upstage passage 임베딩 API. 호출량/비용 집계를 위해 (vecs, n_calls, n_tokens)는
    호출측에서 래핑. 여기선 벡터만 반환."""

    import requests

    url = "https://api.upstage.ai/v1/solar/embeddings"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    out: list[list[float]] = []
    B = 100
    for i in range(0, len(texts), B):
        chunk = texts[i : i + B]
        r = requests.post(url, headers=headers, json={"model": model, "input": chunk}, timeout=60)
        r.raise_for_status()
        data = r.json()["data"]
        out.extend(d["embedding"] for d in sorted(data, key=lambda d: d["index"]))
    return l2_normalize(np.array(out, dtype=np.float32))


# ---------------------------------------------------------------- 클러스터링
def parse_hours(ts: str) -> float:
    """ISO 시각 → 시(hour) 단위 실수(시간 창 비교용)."""
    from datetime import datetime

    return datetime.fromisoformat(ts.replace("Z", "+00:00")).timestamp() / 3600.0


def cluster_online_centroid(
    ids: list[str], vecs: np.ndarray, times_h: list[float], threshold: float, window_h: float
) -> dict[str, int]:
    """온라인 centroid 매칭 (시간순 스트리밍 휴리스틱).

    기사를 발행 시간순으로 처리하며, 활성 클러스터 중 (cosine>=threshold)이고 시간창
    안(현재기사 시각 - 클러스터의 마지막 기사 시각 <= window_h)인 최고 유사 클러스터에
    합류시킨다. 없으면 새 클러스터를 만든다.

    ⚠️ 시간창은 "클러스터의 **마지막** 기사 시각" 기준이다(sliding inactivity window):
    관련 기사가 계속 유입되면 클러스터가 오래 유지된다. 최초 기사 기준 최대 지속시간
    제한이 아니다(#9). 운영 클러스터링과 동일한 증분 방식을 모사한다.
    """

    order = sorted(range(len(ids)), key=lambda i: times_h[i])
    centroids: list[np.ndarray] = []
    csizes: list[int] = []
    ctime: list[float] = []
    labels = [-1] * len(ids)
    for i in order:
        v = vecs[i]
        best, best_sim = -1, threshold
        for c in range(len(centroids)):
            if times_h[i] - ctime[c] > window_h:
                continue
            sim = float(np.dot(v, centroids[c]))  # 정규화됨 → dot=cosine
            if sim >= best_sim:
                best_sim, best = sim, c
        if best == -1:
            centroids.append(v.copy())
            csizes.append(1)
            ctime.append(times_h[i])
            labels[i] = len(centroids) - 1
        else:
            n = csizes[best]
            centroids[best] = (centroids[best] * n + v) / (n + 1)
            centroids[best] /= np.linalg.norm(centroids[best]) or 1.0
            csizes[best] += 1
            ctime[best] = max(ctime[best], times_h[i])
            labels[i] = best
    return {ids[i]: labels[i] for i in range(len(ids))}


def cluster_agglomerative(ids: list[str], vecs: np.ndarray, threshold: float) -> dict[str, int]:
    from sklearn.cluster import AgglomerativeClustering

    if len(ids) == 1:
        return {ids[0]: 0}
    model = AgglomerativeClustering(
        n_clusters=None,
        metric="cosine",
        linkage="average",
        distance_threshold=1.0 - threshold,
    )
    labels = model.fit_predict(vecs)
    return {ids[i]: int(labels[i]) for i in range(len(ids))}


def knn_edges(vecs: np.ndarray, k: int, edge_threshold: float) -> dict[tuple[int, int], float]:
    """대칭 k-NN 엣지 집합. pair를 (min,max)로 정규화해 비대칭 이웃 관계에서도
    엣지가 누락되지 않게 한다(순서 무관, 결정적).

    반환: {(a,b): weight} — 같은 pair가 양쪽에서 나오면 최대 유사도 사용.
    """

    n = vecs.shape[0]
    sims = vecs @ vecs.T
    kk = min(k, n - 1)
    edge_pairs: dict[tuple[int, int], float] = {}
    for i in range(n):
        # 자기 인덱스를 명시적으로 제거한다. argsort는 동일/동점 벡터에서 self가
        # 첫 번째임을 보장하지 않으므로 [1:kk+1] 슬라이싱은 self-loop를 만들 수 있다(#1).
        order = np.argsort(-sims[i])
        nbr = order[order != i][:kk]
        for j in nbr:
            jj = int(j)
            s = float(sims[i, jj])
            if s >= edge_threshold:
                a, b = (i, jj) if i < jj else (jj, i)
                prev = edge_pairs.get((a, b))
                if prev is None or s > prev:
                    edge_pairs[(a, b)] = s
    return edge_pairs


def _leiden_from_edges(
    n: int, edge_pairs: dict[tuple[int, int], float], resolution: float
) -> list[int]:
    """엣지 집합 → Leiden membership. 엣지가 없으면 모두 singleton."""

    if not edge_pairs:
        return list(range(n))  # 전부 singleton (명시적 분기, #11)

    import igraph as ig
    import leidenalg as la

    # 결정적: pair 정렬 순서로 엣지/가중치 구성
    items = sorted(edge_pairs.items())
    edges = [e for e, _ in items]
    weights = [w for _, w in items]
    g = ig.Graph(n=n, edges=edges)
    g.es["weight"] = weights
    part = la.find_partition(
        g,
        la.RBConfigurationVertexPartition,
        weights="weight",
        resolution_parameter=resolution,
        seed=42,
    )
    return list(part.membership)


def cluster_leiden(
    ids: list[str], vecs: np.ndarray, k: int, edge_threshold: float, resolution: float
) -> dict[str, int]:
    """k-NN 그래프 + Leiden. 코사인>=edge_threshold 엣지만 유지.

    엣지는 대칭 pair로 정규화하므로 기사 입력 순서에 무관하게 동일한 그래프가 나온다.
    엣지가 없으면 모든 기사를 singleton으로 처리한다.

    구현 메모: 매 호출마다 sims = vecs @ vecs.T 를 재계산한다(종목 내부 소규모 전제).
    대규모로 확장 시 종목별 sims/이웃을 사전계산해 재사용해야 한다(#8).
    """

    n = len(ids)
    if n == 1:
        return {ids[0]: 0}
    edge_pairs = knn_edges(vecs, k, edge_threshold)
    labels = _leiden_from_edges(n, edge_pairs, resolution)
    return {ids[i]: int(labels[i]) for i in range(n)}


# ---------------------------------------------------------------- 평가지표
def _clusters_from_labels(labels: dict[str, int]) -> dict[int, set]:
    out = defaultdict(set)
    for k, v in labels.items():
        out[v].add(k)
    return out


def bcubed(pred: dict[str, int], gold: dict[str, int]) -> tuple[float, float, float]:
    """B-cubed precision/recall/F1 (주지표)."""

    items = list(gold.keys())
    gold_c = _clusters_from_labels(gold)
    pred_c = _clusters_from_labels(pred)
    gmap = {i: [c for c, s in gold_c.items() if i in s][0] for i in items}
    pmap = {i: pred[i] for i in items}
    prec, rec = 0.0, 0.0
    for i in items:
        pc = pred_c[pmap[i]]
        gc = gold_c[gmap[i]]
        inter = len(pc & gc)
        prec += inter / len(pc)
        rec += inter / len(gc)
    n = len(items)
    prec, rec = prec / n, rec / n
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1


def pairwise_prf(pred: dict[str, int], gold: dict[str, int]) -> tuple[float, float, float]:
    items = list(gold.keys())
    gl = np.array([gold[it] for it in items])
    pl = np.array([pred[it] for it in items])
    tp = fp = fn = 0
    for a in range(len(items)):
        for b in range(a + 1, len(items)):
            same_g = gl[a] == gl[b]
            same_p = pl[a] == pl[b]
            if same_p and same_g:
                tp += 1
            elif same_p and not same_g:
                fp += 1
            elif not same_p and same_g:
                fn += 1
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return prec, rec, f1


def over_under_merge(pred: dict[str, int], gold: dict[str, int]) -> tuple[float, float]:
    """over-merge: 서로 다른 gold인데 같은 pred로 묶인 쌍 비율.
    under-merge: 같은 gold인데 다른 pred로 쪼개진 쌍 비율."""

    items = list(gold.keys())
    gl = [gold[it] for it in items]
    pl = [pred[it] for it in items]
    diff_g_pairs = same_g_pairs = 0
    over = under = 0
    for a in range(len(items)):
        for b in range(a + 1, len(items)):
            sg = gl[a] == gl[b]
            sp = pl[a] == pl[b]
            if sg:
                same_g_pairs += 1
                if not sp:
                    under += 1
            else:
                diff_g_pairs += 1
                if sp:
                    over += 1
    over_rate = over / diff_g_pairs if diff_g_pairs else 0.0
    under_rate = under / same_g_pairs if same_g_pairs else 0.0
    return over_rate, under_rate


def ari_nmi(pred: dict[str, int], gold: dict[str, int]) -> tuple[float, float]:
    from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

    items = list(gold.keys())
    gl = [gold[it] for it in items]
    pl = [pred[it] for it in items]
    return adjusted_rand_score(gl, pl), normalized_mutual_info_score(gl, pl)


def evaluate(pred: dict[str, int], gold: dict[str, int]) -> dict:
    bp, br, bf = bcubed(pred, gold)
    pp, pr, pf = pairwise_prf(pred, gold)
    over, under = over_under_merge(pred, gold)
    ari, nmi = ari_nmi(pred, gold)
    return {
        "bcubed_precision": bp,
        "bcubed_recall": br,
        "bcubed_f1": bf,
        "pairwise_precision": pp,
        "pairwise_recall": pr,
        "pairwise_f1": pf,
        "over_merge_rate": over,
        "under_merge_rate": under,
        "ari": ari,
        "nmi": nmi,
    }


def evaluate_per_stock(
    pred: dict[str, int], rows: list[dict], eligible_only: bool = True, strict: bool = True
) -> dict:
    """종목별 gold로 평가 후 macro 평균. pred/gold 모두 종목 내에서만 비교.

    strict=True(기본): 어떤 종목의 eligible 기사 중 pred에 없는 것이 있으면 예외.
    클러스터링 구현 오류로 예측이 누락된 종목이 조용히 평가에서 빠져 성능이
    과대평가되는 것을 막는다(#4). 예측은 반드시 평가 대상 전원을 덮어야 한다.
    """

    by_stock = defaultdict(list)
    for r in rows:
        if eligible_only and r.get("evaluation_eligible") != "true":
            continue
        by_stock[r["stock_code"]].append(r)
    per = {}
    skipped: dict[str, int] = {}
    for stock, srows in by_stock.items():
        ids = [r["article_stock_id"] for r in srows]
        gold = {r["article_stock_id"]: r["gold_event_id"] for r in srows}
        gmap = {g: i for i, g in enumerate(sorted(set(gold.values())))}
        gold_i = {k: gmap[v] for k, v in gold.items()}
        missing = [i for i in ids if i not in pred]
        if missing:
            if strict:
                raise ValueError(
                    f"{stock}: 예측 누락 {len(missing)}건 (예: {missing[:5]}). "
                    "클러스터링이 평가 대상 전원을 덮지 않음 — 조용한 제외 대신 실패 처리."
                )
            skipped[stock] = len(missing)
            continue
        pred_i = {i: pred[i] for i in ids}
        per[stock] = evaluate(pred_i, gold_i)
    if not per:
        return {"per_stock": {}, "macro": {}, "skipped_stocks": skipped}
    keys = next(iter(per.values())).keys()
    macro = {k: float(np.mean([per[s][k] for s in per])) for k in keys}
    return {"per_stock": per, "macro": macro, "skipped_stocks": skipped}

## 2. UPSTAGE 키 (없으면 Upstage만 skip)


In [ ]:
import os

UPSTAGE_API_KEY = os.environ.get("UPSTAGE_API_KEY", "")  # 필요시 여기에 직접 입력
from google.colab import userdata

UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
print("Upstage 키:", "있음" if UPSTAGE_API_KEY else "없음 → Upstage 실험 skipped")

Upstage 키: 있음


In [ ]:
# 데이터 로드
import csv


def load_split(name):
    with open(f"{name}.csv", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def eligible(rows):
    return [r for r in rows if r.get("evaluation_eligible") == "true"]


dev, val, test = load_split("development"), load_split("validation"), load_split("test")
print("dev/val/test eligible:", len(eligible(dev)), len(eligible(val)), len(eligible(test)))


# split 간 누수 검사(#3): 같은 article_id / gold_event_id / split_unit_id가 두 split에 걸치면 실패
def leak_check(splits):
    seen = {}
    for name, rows in splits.items():
        for r in rows:
            for field in ("article_id", "gold_event_id", "split_unit_id"):
                key = (field, r[field])
                if key in seen and seen[key] != name:
                    raise AssertionError(f"누수: {field}={r[field]} in {seen[key]} & {name}")
                seen.setdefault(key, name)
    print("누수 검사 통과: article_id/gold_event_id/split_unit_id split 중복 0")


leak_check({"development": dev, "validation": val, "test": test})
CACHE = EmbeddingCache(cache_dir=Path("emb_cache"))

dev/val/test eligible: 670 207 210
누수 검사 통과: article_id/gold_event_id/split_unit_id split 중복 0


## 3. 임베딩 (3모델 × 3입력, 캐시)


In [ ]:
# 재현성: 모델 revision을 commit hash로 고정(#5). 캐시 키에 revision 포함됨.
MODELS = [
    ("upstage/embedding-passage", "api", "upstage"),
    ("BAAI/bge-m3", "5617a9f61b028005a4858fdac845db406aefb181", "st"),
    ("intfloat/multilingual-e5-large-instruct", "274baa43b0e13e37fafa6428dbc7938e62e5c439", "st"),
]
import time

EMBED_META = {}  # (model,input_type) -> truncation/revision/cache 통계


def embed_rows(rows, model, revision, kind, input_type, max_seq_length=None):
    raw = [build_input_text(r, input_type) for r in rows]
    prepared = [format_for_model(t, model) for t in raw]  # 전처리 완료 텍스트로 캐시(#2)
    msl = max_seq_length if max_seq_length else "default"
    keys = [
        CACHE.key(model, revision, input_type, text_sha256(t), max_seq_length=msl) for t in prepared
    ]
    CACHE.load(model)
    miss = [i for i, k in enumerate(keys) if CACHE.get(k) is None]
    meta = {"cache_hits": len(keys) - len(miss), "cache_misses": len(miss), "api_calls": 0}
    if miss:
        mt = [prepared[i] for i in miss]
        if kind == "upstage":
            if not UPSTAGE_API_KEY:
                return None, {"status": "skipped", "reason": "no key"}
            v = embed_upstage(mt, UPSTAGE_API_KEY)
            meta["api_calls"] = (len(mt) + 99) // 100
        else:
            t0 = time.time()
            v, emb_meta = embed_sentence_transformer(
                mt, model, revision, DEVICE, max_seq_length=max_seq_length
            )
            meta["embed_seconds"] = time.time() - t0
            meta.update(emb_meta)  # truncation_rate 등 (#6)
        for j, i in enumerate(miss):
            CACHE.put(keys[i], v[j])
        CACHE.save(model)
    elif kind != "upstage":
        meta.update(
            embedding_meta_only(prepared, model, revision, max_seq_length)
        )  # 100% hit도 meta(#3)
    EMBED_META[(model, input_type)] = meta
    mat = l2_normalize(np.vstack([CACHE.get(k) for k in keys]).astype(np.float32))
    meta["status"] = "ok"
    return mat, meta


print("embed_rows 준비 완료")

embed_rows 준비 완료


## 4. 클러스터링 (종목 내부) + per-stock 평가


In [ ]:
def cluster_per_stock(rows, vecs_by_id, method, params):
    by = {}
    for i, r in enumerate(rows):
        by.setdefault(r["stock_code"], []).append(i)
    out = {}
    off = 0
    for stock, idxs in by.items():
        ids = [rows[i]["article_stock_id"] for i in idxs]
        vecs = np.vstack([vecs_by_id[i] for i in ids])
        times = [parse_hours(rows[i]["published_at"]) for i in idxs]
        if method == "centroid":
            lab = cluster_online_centroid(ids, vecs, times, params["threshold"], params["window_h"])
        elif method == "agglomerative":
            lab = cluster_agglomerative(ids, vecs, params["threshold"])
        elif method == "leiden":
            lab = cluster_leiden(ids, vecs, params["k"], params["edge"], params["resolution"])
        mx = -1
        for i, l in lab.items():
            out[i] = l + off
            mx = max(mx, l)
        off += mx + 1
    return out


print("cluster_per_stock 준비 완료")

cluster_per_stock 준비 완료


## 5. 스윕 그리드 (프롬프트 [5])
- centroid/agglomerative threshold 0.70–0.95 step 0.01
- 시간창 24/72/168h
- Leiden k∈{5,10,20,30}, edge 0.70–0.95, resolution∈{0.5,0.8,1.0,1.2,1.5}


In [ ]:
import itertools

C_THR = [round(x, 2) for x in np.arange(0.70, 0.951, 0.01)]
WINS = [24, 72, 168]
L_K = [5, 10, 20, 30]
L_E = [round(x, 2) for x in np.arange(0.70, 0.951, 0.05)]
L_R = [0.5, 0.8, 1.0, 1.2, 1.5]


def grids(methods):
    g = []
    if "centroid" in methods:
        for t in C_THR:
            for w in WINS:
                g.append(("centroid", {"threshold": t, "window_h": w}))
    if "agglomerative" in methods:
        for t in C_THR:
            g.append(("agglomerative", {"threshold": t}))
    if "leiden" in methods:
        for k, e, r in itertools.product(L_K, L_E, L_R):
            g.append(("leiden", {"k": k, "edge": e, "resolution": r}))
    return g


def run_sweep(rows, vecs_by_id, methods):
    runs = []
    for method, params in grids(methods):
        t0 = time.time()
        try:
            pred = cluster_per_stock(rows, vecs_by_id, method, params)
        except Exception as ex:
            runs.append({"method": method, **params, "error": str(ex)})
            continue
        macro = evaluate_per_stock(pred, rows)["macro"]
        runs.append(
            {
                "method": method,
                **{f"param_{k}": v for k, v in params.items()},
                **{f"macro_{k}": round(v, 4) for k, v in macro.items()},
                "n_pred_clusters": len(set(pred.values())),
                "runtime_sec": round(time.time() - t0, 2),
            }
        )
    return runs


print("run_sweep 준비 완료")

run_sweep 준비 완료


## 6. VALIDATION 스윕 실행 (모델·입력·알고리즘·threshold 선택)


In [ ]:
import pandas as pd

val_e = eligible(val)
all_runs = []
for model, revision, kind in MODELS:
    for it in ["A_title", "B_title_desc", "C_title_desc_body"]:
        vecs, meta = embed_rows(val_e, model, revision, kind, it)
        if meta.get("status") == "skipped":
            all_runs.append(
                {"model": model, "input_type": it, "status": "skipped", "reason": meta["reason"]}
            )
            continue
        vbi = {val_e[i]["article_stock_id"]: vecs[i] for i in range(len(val_e))}
        rs = run_sweep(val_e, vbi, ["centroid", "agglomerative", "leiden"])
        for r in rs:
            r["model"] = model
            r["input_type"] = it
            r["split"] = "validation"
        all_runs += rs
        print(model, it, "→", len(rs), "runs")
Path("results").mkdir(exist_ok=True)
df = pd.DataFrame(all_runs)
df.to_csv("results/all_runs.csv", index=False)
print("saved results/all_runs.csv", df.shape)
df.sort_values("macro_bcubed_f1", ascending=False).head(10)

upstage/embedding-passage A_title → 224 runs
upstage/embedding-passage B_title_desc → 224 runs
upstage/embedding-passage C_title_desc_body → 224 runs


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

intfloat/multilingual-e5-large-instruct B_title_desc → 224 runs


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

intfloat/multilingual-e5-large-instruct C_title_desc_body → 224 runs
saved results/all_runs.csv (2016, 21)


,method,param_threshold,param_window_h,macro_bcubed_precision,macro_bcubed_recall,macro_bcubed_f1,macro_pairwise_precision,macro_pairwise_recall,macro_pairwise_f1,macro_over_merge_rate,...,macro_ari,macro_nmi,n_pred_clusters,runtime_sec,model,input_type,split,param_k,param_edge,param_resolution
909,centroid,0.74,72.0,0.9505,0.9896,0.9693,0.9625,0.9706,0.9646,0.0094,...,0.9580,0.9723,56,0.02,BAAI/bge-m3,B_title_desc,validation,NaN,NaN,NaN
906,centroid,0.73,72.0,0.9459,0.9896,0.9669,0.9616,0.9706,0.9641,0.0097,...,0.9574,0.9705,55,0.02,BAAI/bge-m3,B_title_desc,validation,NaN,NaN,NaN
910,centroid,0.74,168.0,0.9460,0.9896,0.9669,0.9597,0.9706,0.9632,0.0117,...,0.9554,0.9658,55,0.02,BAAI/bge-m3,B_title_desc,validation,NaN,NaN,NaN
1408,centroid,0.91,72.0,0.9602,0.9728,0.9656,0.9765,0.9323,0.9503,0.0060,...,0.9421,0.9706,58,0.21,intfloat/multilingual-e5-large-instruct,A_title,validation,NaN,NaN,NaN
907,centroid,0.73,168.0,0.9413,0.9896,0.9644,0.9589,0.9706,0.9627,0.0120,...,0.9548,0.9641,54,0.02,BAAI/bge-m3,B_title_desc,validation,NaN,NaN,NaN
1409,centroid,0.91,168.0,0.9556,0.9728,0.9632,0.9737,0.9323,0.9489,0.0082,...,0.9395,0.9642,57,0.19,intfloat/multilingual-e5-large-instruct,A_title,validation,NaN,NaN,NaN
1641,centroid,0.94,72.0,0.9458,0.9813,0.9624,0.9612,0.9471,0.9496,0.0092,...,0.9416,0.9701,55,0.02,intfloat/multilingual-e5-large-instruct,B_title_desc,validation,NaN,NaN,NaN
1670,agglomerative,0.94,NaN,0.9619,0.9642,0.9619,0.9873,0.9098,0.9402,0.0050,...,0.9319,0.9651,58,0.02,intfloat/multilingual-e5-large-instruct,B_title_desc,validation,NaN,NaN,NaN
903,centroid,0.72,72.0,0.9259,1.0000,0.9605,0.9291,1.0000,0.9591,0.0202,...,0.9490,0.9641,53,0.02,BAAI/bge-m3,B_title_desc,validation,NaN,NaN,NaN
1405,centroid,0.90,72.0,0.9421,0.9813,0.9605,0.9698,0.9471,0.9545,0.0077,...,0.9474,0.9682,54,0.13,intfloat/multilingual-e5-large-instruct,A_title,validation,NaN,NaN,NaN


## 7. 모델 선정 (B-cubed F1 주지표 + over-merge 낮은 순)
Pairwise F1 단독 선택 금지. B-cubed F1 상위 중 over_merge_rate가 낮은 설정 우선.


In [ ]:
ok = df[df.get("status").isna()] if "status" in df else df
ok = ok.dropna(subset=["macro_bcubed_f1"])
# 주지표: bcubed_f1 내림차순, 동률권에서 over_merge 오름차순
ranked = ok.sort_values(["macro_bcubed_f1", "macro_over_merge_rate"], ascending=[False, True])
ranked.to_csv("results/model_comparison.csv", index=False)
best = ranked.iloc[0].to_dict()
print("=== 선정 설정 ===")
for k in [
    "model",
    "input_type",
    "method",
    "param_threshold",
    "param_window_h",
    "param_k",
    "param_edge",
    "param_resolution",
    "macro_bcubed_f1",
    "macro_pairwise_f1",
    "macro_over_merge_rate",
    "macro_under_merge_rate",
    "macro_ari",
    "macro_nmi",
]:
    if k in best and pd.notna(best[k]):
        print(f"  {k}: {best[k]}")

=== 선정 설정 ===
  model: BAAI/bge-m3
  input_type: B_title_desc
  method: centroid
  param_threshold: 0.74
  param_window_h: 72.0
  macro_bcubed_f1: 0.9693
  macro_pairwise_f1: 0.9646
  macro_over_merge_rate: 0.0094
  macro_under_merge_rate: 0.0294
  macro_ari: 0.958
  macro_nmi: 0.9723


## 8. threshold 탐색 곡선 저장 + 플롯


In [ ]:
import matplotlib.pyplot as plt

Path("plots").mkdir(exist_ok=True)
th = ok[ok["method"].isin(["centroid", "agglomerative"])].copy()
th.to_csv("results/threshold_search.csv", index=False)


# centroid는 window_h가 달라도 같은 선으로 섞이면 안 됨 → window_h까지 그룹화(#7).
# agglomerative는 window_h가 없으므로 그룹키에서 자동 제외(NaN 단일값).
def wtag(v):
    import math

    return "naN" if (v is None or (isinstance(v, float) and math.isnan(v))) else f"w{int(v)}"


th["_w"] = th["param_window_h"].apply(wtag)
for (model, it, method, w), g in th.groupby(["model", "input_type", "method", "_w"]):
    gg = g.sort_values("param_threshold")
    plt.figure(figsize=(6, 4))
    plt.plot(gg["param_threshold"], gg["macro_bcubed_f1"], marker=".", label="bcubed_f1")
    plt.plot(gg["param_threshold"], gg["macro_over_merge_rate"], marker=".", label="over_merge")
    ttl = f"{method} | {model.split('/')[-1]} | {it}" + (f" | {w}" if method == "centroid" else "")
    plt.title(ttl)
    plt.xlabel("threshold")
    plt.legend()
    plt.grid(True)
    fn = f"plots/thr_{method}_{model.split('/')[-1]}_{it}_{w}.png"
    plt.savefig(fn, bbox_inches="tight")
    plt.close()
print("threshold plots saved (centroid: window_h별 분리)")

threshold plots saved (centroid: window_h별 분리)


## 9. TEST 최종 평가 (확정 설정으로 1회만)


In [ ]:
test_e = eligible(test)
bm = best["model"]
bit = best["input_type"]
bmethod = best["method"]
rev = dict((m, r) for m, r, _ in MODELS).get(bm, "main")
kind = dict((m, k) for m, _, k in MODELS)[bm]
vecs, meta = embed_rows(test_e, bm, rev, kind, bit)
vbi = {test_e[i]["article_stock_id"]: vecs[i] for i in range(len(test_e))}
params = {}
for pk in ["threshold", "window_h", "k", "edge", "resolution"]:
    key = f"param_{pk}"
    if key in best and pd.notna(best[key]):
        params[pk] = best[key]
pred = cluster_per_stock(test_e, vbi, bmethod, params)
res = evaluate_per_stock(pred, test_e)
import json as _j

Path("results").mkdir(exist_ok=True)
pd.DataFrame(
    [
        {
            "model": bm,
            "input_type": bit,
            "method": bmethod,
            **{f"param_{k}": v for k, v in params.items()},
            **{f"macro_{k}": round(v, 4) for k, v in res["macro"].items()},
        }
    ]
).to_csv("results/final_test_result.csv", index=False)
print("=== TEST 최종 ===")
print(_j.dumps(res["macro"], indent=2, ensure_ascii=False))

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

=== TEST 최종 ===
{
  "bcubed_precision": 0.9416494368652953,
  "bcubed_recall": 0.9497229672785348,
  "bcubed_f1": 0.9448168723658432,
  "pairwise_precision": 0.9047699133559789,
  "pairwise_recall": 0.9422993685623314,
  "pairwise_f1": 0.9204470794261427,
  "over_merge_rate": 0.013722998078089226,
  "under_merge_rate": 0.05770063143766864,
  "ari": 0.9098648966360712,
  "nmi": 0.9596475177534629
}


## 10. over-merge / under-merge 사례 저장


In [ ]:
# 확정 설정(test) 예측의 오류 쌍. 분석용 컬럼 확장(#12). 종목 내부 쌍만 순회.
def error_cases(pred, rows, vbi):
    rid = {r["article_stock_id"]: r for r in rows}
    by = {}
    for r in rows:
        by.setdefault(r["stock_code"], []).append(r["article_stock_id"])
    cols = [
        "stock_code",
        "id_a",
        "id_b",
        "published_a",
        "published_b",
        "gold_a",
        "gold_b",
        "pred_a",
        "pred_b",
        "cosine",
        "title_a",
        "title_b",
        "desc_a",
        "desc_b",
        "error_type",
    ]
    over = []
    under = []
    for stock, ids in by.items():
        for x in range(len(ids)):
            for y in range(x + 1, len(ids)):
                ia, ib = ids[x], ids[y]
                ra, rb = rid[ia], rid[ib]
                sg = ra["gold_event_id"] == rb["gold_event_id"]
                sp = pred[ia] == pred[ib]
                if sg == sp:
                    continue
                cos = float(np.dot(vbi[ia], vbi[ib]))
                row = [
                    stock,
                    ia,
                    ib,
                    ra["published_at"],
                    rb["published_at"],
                    ra["gold_event_id"],
                    rb["gold_event_id"],
                    pred[ia],
                    pred[ib],
                    round(cos, 4),
                    ra["title"],
                    rb["title"],
                    ra.get("description", "")[:80],
                    rb.get("description", "")[:80],
                ]
                if sp and not sg:
                    over.append(row + ["over_merge"])
                else:
                    under.append(row + ["under_merge"])
    return cols, over, under


cols, ov, un = error_cases(pred, test_e, vbi)
CFG = f"model={bm}|input={bit}|method={bmethod}|params={params}"
pd.DataFrame(ov, columns=cols).assign(config=CFG).to_csv(
    "results/over_merge_cases.csv", index=False
)
pd.DataFrame(un, columns=cols).assign(config=CFG).to_csv(
    "results/under_merge_cases.csv", index=False
)
print("over-merge 쌍:", len(ov), " under-merge 쌍:", len(un), " | config:", CFG)

over-merge 쌍: 46  under-merge 쌍: 28  | config: model=BAAI/bge-m3|input=B_title_desc|method=centroid|params={'threshold': 0.74, 'window_h': 72.0}


## 11. 방법 4: embedding 후보 + bge-reranker-v2-m3 + Leiden (선택적 확장)
**정식 스윗 비교 대상이 아니라 선택적 확장 실험**입니다. 3가지 기본 방법을 validation에서 비교해 최종 설정을 고르고, reranker+Leiden은 참고용으로 **validation에서만** 시연 평가합니다(test는 이미 확정 설정 1회로 소진 → test leakage 방지, #2). 후보 top-k 쌍에만 reranker를 적용합니다.


In [ ]:
# reranker+Leiden: validation에서만 시연(정식 선정/ test 미사용). 엣지는 대칭 정규화(#1).
try:
    from FlagEmbedding import FlagReranker

    reranker = FlagReranker(
        "BAAI/bge-reranker-v2-m3",
        revision="953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
        use_fp16=(DEVICE == "cuda"),
    )

    def cluster_rerank_leiden(rows, vecs_by_id, input_type, k=10, resolution=1.0, rerank_thr=0.5):
        by = {}
        for i, r in enumerate(rows):
            by.setdefault(r["stock_code"], []).append(i)
        out = {}
        off = 0
        for stock, idxs in by.items():
            ids = [rows[i]["article_stock_id"] for i in idxs]
            texts = [build_input_text(rows[i], input_type) for i in idxs]
            vecs = np.vstack([vecs_by_id[i] for i in ids])
            sims = vecs @ vecs.T
            n = len(ids)
            kk = min(k, n - 1)
            # embedding top-k 후보 쌍을 (min,max)로 정규화 수집(비대칭/중복/순서 무관, #1).
            # 자기 인덱스를 명시적으로 제거해 self-loop 없이 실제 이웃 k개를 확보(#1-self).
            cand = set()
            for i in range(n):
                order = np.argsort(-sims[i])
                nbr = order[order != i][:kk]
                for j in nbr:
                    a, b = (i, int(j)) if i < int(j) else (int(j), i)
                    cand.add((a, b))
            edge_pairs = {}
            if cand:
                pairs = sorted(cand)
                scores = reranker.compute_score(
                    [[texts[a], texts[b]] for a, b in pairs], normalize=True
                )
                if not isinstance(scores, list):
                    scores = [scores]
                for (a, b), s in zip(pairs, scores):
                    if s >= rerank_thr:
                        edge_pairs[(a, b)] = float(s)
            labels = _leiden_from_edges(n, edge_pairs, resolution)  # 엣지 없으면 singleton(#11)
            mx = -1
            for i, l in enumerate(labels):
                out[ids[i]] = l + off
                mx = max(mx, l)
            off += mx + 1
        return out

    # validation에서 최상위 임베딩 설정으로 시연 (best model/input 재사용)
    val_vecs, _ = embed_rows(
        val_e,
        best["model"],
        dict((m, r) for m, r, _ in MODELS)[best["model"]],
        dict((m, k) for m, _, k in MODELS)[best["model"]],
        best["input_type"],
    )
    val_vbi = {val_e[i]["article_stock_id"]: val_vecs[i] for i in range(len(val_e))}
    predR = cluster_rerank_leiden(val_e, val_vbi, best["input_type"])
    resR = evaluate_per_stock(predR, val_e)["macro"]
    pd.DataFrame(
        [
            {
                "method": "rerank_leiden",
                "split": "validation",
                "model": best["model"],
                "input_type": best["input_type"],
                **{f"macro_{k}": round(v, 4) for k, v in resR.items()},
            }
        ]
    ).to_csv("results/rerank_leiden_validation.csv", index=False)
    print("reranker+Leiden (validation, 참고용):", {k: round(v, 3) for k, v in resR.items()})
except Exception as e:
    print("reranker 방법 skip:", e)

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Compute Scores: 100%|██████████| 3/3 [00:03<00:00,  1.12s/it]


reranker+Leiden (validation, 참고용): {'bcubed_precision': 0.799, 'bcubed_recall': 0.817, 'bcubed_f1': 0.801, 'pairwise_precision': 0.738, 'pairwise_recall': 0.66, 'pairwise_f1': 0.678, 'over_merge_rate': 0.068, 'under_merge_rate': 0.34, 'ari': 0.591, 'nmi': 0.791}


## 12. 산출물 요약

- `results/all_runs.csv` — 전체 스윕
- `results/model_comparison.csv` — B-cubed F1 + over-merge 순위
- `results/threshold_search.csv` — threshold 곡선
- `results/final_test_result.csv` — test 최종 1회
- `results/over_merge_cases.csv`, `under_merge_cases.csv`
- `plots/*.png`

> 마지막에 왼쪽 파일 탭에서 `results/`, `plots/` 폴더를 압축해 내려받으세요:
> `!zip -r experiment_outputs.zip results plots` 후 다운로드.


In [ ]:
!zip -qr experiment_outputs.zip results plots && echo "experiment_outputs.zip 생성 완료 (좌측 파일탭에서 다운로드)"

experiment_outputs.zip 생성 완료 (좌측 파일탭에서 다운로드)
